# Food Price CPI — 3-Month Forecast Experiment

This notebook runs the 3-month food CPI forecasting experiment — a companion
to the 18-month CFPR-replica.  The shorter horizon provides a denser eval
sample in recent history, making model selection and calibration assessment
more statistically reliable.

**Task:** Forecast Canadian food CPI index values 3 months ahead across 9 food categories.  
**Predictors compared:**
- `LastValuePredictor` — naive baseline (performance floor)
- `DartsAutoARIMAPredictor` — univariate AutoARIMA
- `DartsAutoARIMAPredictor` with FRED covariates

**Primary metric:** CRPS (lower is better).  
**Supplementary metric:** MAPE on median forecast.  

## Why Two Horizons?

| | 18-month | 3-month |
|--|---------|--------|
| **Eval origins** | ~5 | ~9 |
| **Eval CRPS reliability** | Lower (more variance) | Higher (less variance) |
| **Method discrimination** | Harder (horizon uncertainty dominates) | Easier |
| **Practical relevance** | Matches CFPR reporting cycle | Near-term planning horizon |

The 3m experiment is particularly useful for:
- **Model selection:** more eval data → more confident ranking of predictors
- **Calibration:** more quantile-coverage data points per category
- **FRED covariate relevance:** shorter-horizon models may respond differently to macro indicators

## Sections
1. Setup: register data and load specs
2. Run multi-target backtest
3. CRPS comparison table
4. MAPE comparison
5. Run multi-target eval (protected window)
6. Eval density discussion

## Prerequisites

```bash
uv run python scripts/fetch_cpi.py
uv run python scripts/fetch_fred.py   # requires FRED_API_KEY in .env
```

In [ ]:
import sys
from pathlib import Path

import yaml

repo_root = Path().resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Setup: Register Data and Load Specs

In [ ]:
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import FREDAdapter, StatCanAdapter
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    MultiTargetEvalSpec,
    multi_backtest,
    multi_evaluate,
)
from aieng.forecasting.evaluation.eval import EvalTracker
from methods.darts_arima import DartsAutoARIMAPredictor
from methods.naive import LastValuePredictor

load_dotenv(repo_root / ".env")

CPI_TABLE_ID = "18-10-0004-11"
CACHE_DIR = repo_root / "data" / "statcan"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

svc = DataService()

FOOD_CPI_SERIES = [
    ("cpi_food_canada", "Food"),
    ("cpi_bakery_cereal_canada", "Bakery and cereal products (excluding baby food)"),
    ("cpi_dairy_eggs_canada", "Dairy products and eggs"),
    ("cpi_fish_seafood_canada", "Fish, seafood and other marine products"),
    ("cpi_restaurants_canada", "Food purchased from restaurants"),
    ("cpi_fruit_preparations_nuts_canada", "Fruit, fruit preparations and nuts"),
    ("cpi_meat_canada", "Meat"),
    ("cpi_other_food_nonalcoholic_canada", "Other food products and non-alcoholic beverages"),
    ("cpi_vegetables_preparations_canada", "Vegetables and vegetable preparations"),
]

for series_id, product_group in FOOD_CPI_SERIES:
    svc.register(
        series_id,
        StatCanAdapter(
            table_id=CPI_TABLE_ID,
            member_filter={"GEO": "Canada", "Products and product groups": product_group},
            cache_dir=CACHE_DIR,
        ),
        SeriesMetadata(
            series_id=series_id,
            description=f"CPI {product_group}, Canada (2002=100)",
            source="StatCan",
            units="Index 2002=100",
            frequency="MS",
            table_id=CPI_TABLE_ID,
        ),
    )

FRED_SERIES = [
    ("fred_us_cpi_food_at_home", "CPIFABSL", "US CPI: Food at Home", "Index 1982-84=100"),
    ("fred_us_cpi_meats_poultry_fish_eggs", "CUSR0000SAF112", "US CPI: Meats/Poultry/Fish/Eggs", "Index 1982-84=100"),
    ("fred_us_cpi_fruits_vegetables", "CUSR0000SAF113", "US CPI: Fruits & Vegetables", "Index 1982-84=100"),
    ("fred_canada_10yr_bond_yield", "IRLTLT01CAM156N", "Canada 10yr Bond Yield", "Percent"),
    ("fred_canada_us_exchange_rate", "EXCAUS", "Canada/US Exchange Rate", "CAD per USD"),
    ("fred_sp100_volatility_vxo", "VXOCLS", "S&P 100 Volatility (VXO)", "Index"),
    ("fred_wilshire_5000", "WILL5000IND", "Wilshire 5000 Index", "Index"),
]

fred_available = []
for series_id, fred_id, description, units in FRED_SERIES:
    try:
        svc.register(
            series_id,
            FREDAdapter(fred_id),
            SeriesMetadata(
                series_id=series_id,
                description=description,
                source=f"FRED ({fred_id})",
                units=units,
                frequency="MS",
            ),
        )
        fred_available.append(series_id)
    except Exception as e:
        print(f"  [WARN] {series_id}: {e}")

print(f"Registered: {len(FOOD_CPI_SERIES)} food CPI + {len(fred_available)} FRED series.")

In [ ]:
spec_path = repo_root / "reference_specs" / "food_cpi" / "food_cpi_3m_backtest.yaml"
with spec_path.open() as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(f"Backtest spec: {len(backtest_spec.tasks)} tasks")
print(f"Window: {backtest_spec.start.date()} → {backtest_spec.end.date()}")
print(f"Stride: {backtest_spec.stride} months (quarterly origins)")
print(f"Candidate origins: {len(backtest_spec.specs()[0].origins())}")

In [ ]:
predictors = [
    LastValuePredictor(),
    DartsAutoARIMAPredictor(num_samples=500),
]
if fred_available:
    predictors.append(DartsAutoARIMAPredictor(num_samples=500, covariate_series_ids=fred_available))

for p in predictors:
    print(f"  {p.predictor_id}")

## 2. Run Multi-Target Backtest

In [ ]:
import time

all_results = {}

for predictor in predictors:
    print(f"Running backtest: {predictor.predictor_id} ...")
    t0 = time.time()
    results = multi_backtest(predictor=predictor, spec=backtest_spec, data_service=svc)
    elapsed = time.time() - t0
    mean_crps = np.mean([r.mean_crps for r in results.values()])
    print(f"  Done in {elapsed:.0f}s — mean CRPS: {mean_crps:.4f}")
    all_results[predictor.predictor_id] = results

## 3. CRPS Comparison Table

In [ ]:
CATEGORY_LABELS = {
    "food_cpi_overall_3m": "Overall Food",
    "food_cpi_bakery_cereal_3m": "Bakery & Cereal",
    "food_cpi_dairy_eggs_3m": "Dairy & Eggs",
    "food_cpi_fish_seafood_3m": "Fish & Seafood",
    "food_cpi_restaurants_3m": "Restaurants",
    "food_cpi_fruit_preparations_nuts_3m": "Fruit & Nuts",
    "food_cpi_meat_3m": "Meat",
    "food_cpi_other_food_3m": "Other Food",
    "food_cpi_vegetables_3m": "Vegetables",
}

crps_data = {
    predictor_id: {CATEGORY_LABELS.get(tid, tid): round(r.mean_crps, 4) for tid, r in task_results.items()}
    for predictor_id, task_results in all_results.items()
}

crps_df = pd.DataFrame(crps_data)
crps_df.loc["Mean"] = crps_df.mean().round(4)
crps_df

## 4. MAPE Comparison

In [ ]:
as_of_now = datetime.now(tz=timezone.utc).replace(tzinfo=None)


def compute_mape(task_results: dict) -> dict[str, float]:
    mapes = {}
    for task_id, result in task_results.items():
        target_series_id = result.spec.task.target_series_id
        full_series = svc.get_series(target_series_id, as_of=as_of_now)
        full_series["timestamp"] = pd.to_datetime(full_series["timestamp"])
        full_series = full_series.set_index("timestamp")["value"]
        ape_list = []
        for pred in result.predictions:
            ts = pd.Timestamp(pred.forecast_date)
            if ts in full_series.index:
                actual = full_series[ts]
                if actual != 0:
                    ape_list.append(abs(pred.payload.point_forecast - actual) / abs(actual) * 100)
        mapes[CATEGORY_LABELS.get(task_id, task_id)] = round(float(np.mean(ape_list)), 3) if ape_list else float("nan")
    return mapes


mape_df = pd.DataFrame({pid: compute_mape(tr) for pid, tr in all_results.items()})
mape_df.loc["Mean"] = mape_df.mean().round(3)
mape_df

## 5. Multi-Target Eval (Protected Window)

In [ ]:
eval_spec_path = repo_root / "reference_specs" / "food_cpi" / "food_cpi_3m_eval.yaml"
with eval_spec_path.open() as f:
    eval_spec = MultiTargetEvalSpec.model_validate(yaml.safe_load(f))

print(f"Eval spec: {eval_spec.spec_id}")
print(f"Window: {eval_spec.start.date()} → {eval_spec.end.date()}")
print(f"Max runs: {eval_spec.max_runs}")

tracker = EvalTracker(Path("eval_runs_3m.yaml"))
print(f"Runs used: {tracker.runs_for(eval_spec.spec_id)} / {eval_spec.max_runs}")

In [ ]:
# Uncomment to run eval — costs one budget run.

# best_predictor = DartsAutoARIMAPredictor(num_samples=500)
# eval_results = multi_evaluate(
#     predictor=best_predictor,
#     spec=eval_spec,
#     data_service=svc,
#     tracker=tracker,
# )
# eval_crps = {CATEGORY_LABELS.get(tid, tid): round(r.mean_crps, 4) for tid, r in eval_results.items()}
# eval_series = pd.Series(eval_crps, name="eval_mean_crps")
# print(f"Mean CRPS: {eval_series.mean():.4f}")
# eval_series
print("Eval cell ready — uncomment to consume one budget run.")

## 6. Eval Density: 3-Month vs. 18-Month Comparison

This section illustrates why the 3-month experiment provides more reliable eval estimates.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Show origin timelines for both eval specs side by side.
eval_18m_path = repo_root / "reference_specs" / "food_cpi" / "food_cpi_18m_eval.yaml"
with eval_18m_path.open() as f:
    eval_18m_spec = MultiTargetEvalSpec.model_validate(yaml.safe_load(f))

# Origins for a single task (same window params for all 9 categories)
single_18m = eval_18m_spec.specs()[0]
single_3m = eval_spec.specs()[0]

origins_18m = single_18m.origins()
origins_3m = single_3m.origins()

fig, ax = plt.subplots(figsize=(12, 3))

ax.scatter(
    origins_18m,
    [1.1] * len(origins_18m),
    color="#2563eb",
    s=80,
    zorder=3,
    label=f"18m eval origins ({len(origins_18m)})",
    marker="D",
)
ax.scatter(
    origins_3m,
    [0.9] * len(origins_3m),
    color="#dc2626",
    s=80,
    zorder=3,
    label=f"3m eval origins ({len(origins_3m)})",
    marker="o",
)

ax.set_yticks([0.9, 1.1])
ax.set_yticklabels(["3-month eval", "18-month eval"])
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
ax.set_ylim(0.7, 1.3)
ax.set_title("Eval Origin Timeline: 3-Month vs. 18-Month Experiments", fontsize=12, fontweight="bold")
ax.legend()
fig.tight_layout()
plt.show()

print(f"\n18-month eval: {len(origins_18m)} origins")
print(f"3-month eval:  {len(origins_3m)} origins")
print(f"\nMore origins → lower variance in mean CRPS estimate → more confident model ranking.")

In [ ]:
# Backtest CRPS per origin for Overall Food — shows how variance in per-origin scores
# translates into uncertainty in the mean eval CRPS estimate.
task_id = "food_cpi_overall_3m"
result = all_results.get("darts_autoarima", {}).get(task_id)

if result:
    scores = np.array(result.scores)
    n_eval_3m = len(origins_3m)
    n_eval_18m = len(origins_18m)

    se_3m = scores.std() / np.sqrt(n_eval_3m)
    se_18m = scores.std() / np.sqrt(n_eval_18m)

    print(f"Overall Food CRPS standard deviation (backtest): {scores.std():.4f}")
    print()
    print(f"Estimated SE of mean CRPS with {n_eval_3m} eval origins (3m):  ±{se_3m:.4f}")
    print(f"Estimated SE of mean CRPS with {n_eval_18m} eval origins (18m): ±{se_18m:.4f}")
    print()
    print(f"3m eval provides {se_18m / se_3m:.1f}x tighter confidence on mean CRPS.")